In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from time import sleep
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from scipy.special import softmax

In [2]:
rawdata = '../rawdata/'
preprocesseddata = '../preprocesseddata'

In [3]:
tesla_1 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_Tesla OR Model 3 OR Model3 OR Model Y OR Model X OR Tesla Y OR Tesla X.csv')
# tesla_2 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_Model Y OR Model X OR Tesla Y OR Tesla X.csv')
audi_1 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_Audi Q4 E-Tron OR e-Tron OR etron.csv')
audi_2 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_e-Tron OR etron OR Audi EV OR Audi electric.csv')
bmw = pd.read_csv(rawdata+'2020-01-01_2023-04-10_BMW i3 OR BMW i4 OR BMW EV OR BMW Electric.csv')
mbenz_1 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_EQB OR EQS OR mercedes EV OR mercedes electric OR mbenz ev.csv')
mbenz_2 = pd.read_csv(rawdata+'2020-01-01_2023-04-10_EQB OR EQS OR mercedes EV OR mercedes electric.csv')
vw = pd.read_csv(rawdata+'2020-01-01_2023-04-10_Volkswagen ID.4 OR Volkswagen ID 4 OR Volkswagen ID-4 OR Volkswagen ID4.csv')

In [4]:
df = pd.concat([tesla_1, 
                # tesla_2, 
                audi_1, audi_2, bmw, mbenz_1, mbenz_2, vw], axis=0)
df.head()

,Datetime,Tweet Id,Text,Username
0,2023-04-09 23:44:47+00:00,1645211184512266240,@hamids @Rivian @Tesla Definitely. R1S is a mu...,vRACEuWIN
1,2023-04-09 23:32:34+00:00,1645208110309720064,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",ShoppingAlertAU
2,2023-04-09 23:26:38+00:00,1645206616491302912,@hamids @Rivian @Tesla The only problem is RIV...,goldentouch73
3,2023-04-09 23:23:56+00:00,1645205937244852224,"Model X Long Range Demo\nKing Of Prussia, PA\n...",tesla_inventory
4,2023-04-09 23:18:28+00:00,1645204561156767744,1074812-00-B New Set of (4) TPMS Tire Pressure...,happy0shopping


In [5]:
df.shape

(260075, 4)

#### Dropping Duplicates Rows

In [6]:
df.drop_duplicates(subset=['Text'], keep='first', inplace=True, ignore_index=True)

In [7]:
df.shape

(244228, 4)

### Pre-processing the data

#### Removing mentions and links

In [8]:
tweet = []
tmpdf = df.Text.str.split(' ')
with tqdm(total=len(df['Text'])) as pbar:
    for i in range(len(tmpdf)):
        tmpList = ['@user' if word.startswith('@') else word for word in tmpdf[i]]
        tmpList = ['http' if word.startswith('http') else word for word in tmpList]
        tweet.append(' '.join(tmpList))
        pbar.update(1)
df['Text'] = tweet

  0%|          | 0/244228 [00:00<?, ?it/s]

In [9]:
df.head()

,Datetime,Tweet Id,Text,Username
0,2023-04-09 23:44:47+00:00,1645211184512266240,@user @user @user Definitely. R1S is a much be...,vRACEuWIN
1,2023-04-09 23:32:34+00:00,1645208110309720064,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",ShoppingAlertAU
2,2023-04-09 23:26:38+00:00,1645206616491302912,@user @user @user The only problem is RIVN kee...,goldentouch73
3,2023-04-09 23:23:56+00:00,1645205937244852224,"Model X Long Range Demo\nKing Of Prussia, PA\n...",tesla_inventory
4,2023-04-09 23:18:28+00:00,1645204561156767744,1074812-00-B New Set of (4) TPMS Tire Pressure...,happy0shopping


#### Dropping unnecessary rows

In [10]:
# List of the makes and models that data were scraped off of:

cars = [
    'tesla', 'model 3', 'model3', 'model x', 'model x', 'tesla y', 'tesla x',
    'audi q4 e-tron', 'e-tron', 'etron',
    'e-tron', 'etron', 'audi ev', 'audi electric',
    'bmw i3', 'bmw i4', 'bmw ev', 'bmw electric',
    'eqb', 'eqs', 'mercedes ev', 'mercedes electric', 'mbenz ev',
    'volkswagen id.4',' volkswagen id 4', 'volkswagen id-4', 'volkswagen id4',
    'rivian', 'r1s', 'r1t', 'taycan'
]

In [11]:
# Keeping only tweets that have the desired words in their text. 
#(In the first place, tweets with the words in their username and text were scraped)

# A function that finds the cars in a given text
def find_cars(tweet):
    for car in cars:
        if car in tweet.lower():
            return car
    return None

# Applying the function on the data
df['Word'] = df['Text'].apply(find_cars)
df = df[df['Word'].notnull()]
df = df[['Text', 'Datetime', 'Word']]

In [12]:
df

,Text,Datetime,Word
0,@user @user @user Definitely. R1S is a much be...,2023-04-09 23:44:47+00:00,model x
1,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",2023-04-09 23:32:34+00:00,tesla
2,@user @user @user The only problem is RIVN kee...,2023-04-09 23:26:38+00:00,tesla
3,"Model X Long Range Demo\nKing Of Prussia, PA\n...",2023-04-09 23:23:56+00:00,model x
4,1074812-00-B New Set of (4) TPMS Tire Pressure...,2023-04-09 23:18:28+00:00,tesla
...,...,...,...
244203,Main Volkswagen ID 4 Forums • Sporty VW ID4 Mo...,2020-05-02 07:55:09+00:00,volkswagen id 4
244204,2021/22 Volkswagen ID.4 GTX Coupe spied testin...,2020-04-30 18:36:33+00:00,volkswagen id.4
244216,Here's a better look 👀 at the upcoming Volkswa...,2020-03-03 18:31:14+00:00,volkswagen id.4
244225,Here’s an exclusive first look at the Volkswag...,2020-03-03 09:30:00+00:00,volkswagen id.4


#### Adding Make and Models of the cars

In [13]:
# define the dictionary of make and model mappings

make_model_dict = {
    'tesla': {'make': 'Tesla', 'model': 'not mentioned'},
    'model 3': {'make': 'Tesla', 'model': 'Model 3'}, 
    'model3': {'make': 'Tesla', 'model': 'Model 3'},
    'model y': {'make': 'Tesla', 'model': 'Model Y'},
    'model x': {'make': 'Tesla', 'model': 'Model X'},
    'tesla y': {'make': 'Tesla', 'model': 'Model Y'},
    'tesla x': {'make': 'Tesla', 'model': 'Model X'},
    'volkswagen id.4': {'make': 'Volkswagen', 'model': 'ID.4'},
    'volkswagen id 4': {'make': 'Volkswagen', 'model': 'ID.4'},
    'volkswagen id-4': {'make': 'Volkswagen', 'model': 'ID.4'},
    'volkswagen id4': {'make': 'Volkswagen', 'model': 'ID.4'},
    'audi q4 e-tron': {'make': 'Audi', 'model': 'Q4 E-Tron'},
    'e-tron': {'make': 'Audi', 'model': 'E-Tron'},
    'etron': {'make': 'Audi', 'model': 'E-Tron'},
    'audi ev': {'make': 'Audi', 'model': 'not mentioned'},
    'audi electric': {'make': 'Audi', 'model': 'not mentioned'},
    'bmw ev': {'make': 'BMW', 'model': 'not mentioned'},
    'bmw electric': {'make': 'BMW', 'model': 'not mentioned'},
    'bmw i3': {'make': 'BMW', 'model': 'i3'},
    'bmw i4': {'make': 'BMW', 'model': 'i4'},
    'mercedes ev': {'make': 'Mercedes-Benz', 'model': 'not mentioned'},
    'mercedes electric': {'make': 'Mercedes-Benz', 'model': 'not mentioned'},
    'mbenz ev': {'make': 'Mercedes-Benz', 'model': 'not mentioned'},
    'eqb': {'make': 'Mercedes-Benz', 'model': 'EQB'},
    'eqs': {'make': 'Mercedes-Benz', 'model': 'EQS'},
    'rivian': {'make': 'Rivian', 'model': 'not mentioned'},
    'r1s': {'make': 'Rivian', 'model': 'r1s'},
    'r1t': {'make': 'Rivian', 'model': 'r1t'},
    'taycan':{'make': 'Porsche', 'model': 'taycan'}
}

# iterate through the DataFrame and update the 'Make' and 'Model' columns
for idx, row in df.iterrows():
    for key, value in make_model_dict.items():
        if key in row['Word']:
            df.at[idx, 'Make'] = value['make']
            df.at[idx, 'Model'] = value['model']
            break

In [14]:
df

,Text,Datetime,Word,Make,Model
0,@user @user @user Definitely. R1S is a much be...,2023-04-09 23:44:47+00:00,model x,Tesla,Model X
1,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",2023-04-09 23:32:34+00:00,tesla,Tesla,not mentioned
2,@user @user @user The only problem is RIVN kee...,2023-04-09 23:26:38+00:00,tesla,Tesla,not mentioned
3,"Model X Long Range Demo\nKing Of Prussia, PA\n...",2023-04-09 23:23:56+00:00,model x,Tesla,Model X
4,1074812-00-B New Set of (4) TPMS Tire Pressure...,2023-04-09 23:18:28+00:00,tesla,Tesla,not mentioned
...,...,...,...,...,...
244203,Main Volkswagen ID 4 Forums • Sporty VW ID4 Mo...,2020-05-02 07:55:09+00:00,volkswagen id 4,Volkswagen,ID.4
244204,2021/22 Volkswagen ID.4 GTX Coupe spied testin...,2020-04-30 18:36:33+00:00,volkswagen id.4,Volkswagen,ID.4
244216,Here's a better look 👀 at the upcoming Volkswa...,2020-03-03 18:31:14+00:00,volkswagen id.4,Volkswagen,ID.4
244225,Here’s an exclusive first look at the Volkswag...,2020-03-03 09:30:00+00:00,volkswagen id.4,Volkswagen,ID.4


#### Dropping unrelated tweets

In [15]:
# Looking only through rows with the word tesla and unmentioned models
df_tesla = df[(df['Make'] == 'Tesla') & (df['Model'] == 'not mentioned')]

In [16]:
df_tesla

,Text,Datetime,Word,Make,Model
1,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",2023-04-09 23:32:34+00:00,tesla,Tesla,not mentioned
2,@user @user @user The only problem is RIVN kee...,2023-04-09 23:26:38+00:00,tesla,Tesla,not mentioned
4,1074812-00-B New Set of (4) TPMS Tire Pressure...,2023-04-09 23:18:28+00:00,tesla,Tesla,not mentioned
9,@user @user @user My guests have great experie...,2023-04-09 23:02:19+00:00,tesla,Tesla,not mentioned
10,Tesla Model X Sinks In Danube River And Sells ...,2023-04-09 23:01:30+00:00,tesla,Tesla,not mentioned
...,...,...,...,...,...
244154,Watch Tesla CEO Elon Musk test drive VW ID.3 -...,2020-09-07 13:46:02+00:00,tesla,Tesla,not mentioned
244164,Is It Tesla Enough For You? Volkswagen Reveals...,2020-09-04 17:20:31+00:00,tesla,Tesla,not mentioned
244167,ID.4 Will Not Rival the Tesla Model Y #ev #ele...,2020-08-31 15:10:40+00:00,tesla,Tesla,not mentioned
244184,Volkswagen's ID.4 EV Crossover To Arrive In De...,2020-08-07 14:19:42+00:00,tesla,Tesla,not mentioned


In [17]:
related_words = ['car', 'cars', 'drive', 'driving', 'model', ' model s', 'models', 'roadster', 'ev', 'electric',
                 'supercharger', 'autopilot', 'ludicrous mode', 'full self-driving', 'fsd',
                 'vehicle', 'vehicles', 'battery', 'charging', 'range', 'acceleration', 'performance',
                 'efficiency', 'transportation', 'sustainable', 'carbon emissions', 'fuel',
                 'wheel', 'wheels', 'tire', 'tires', 'autonomy', 'speed', 'luxury', 'torque',
                 'braking', 'brake', 'sedan', 'crossover', 'suv', 'hatchback', 'accident', 'feature'
                 'safety', 'price', 'park', 'color', 'hybrid', 'door', 'ring', 'motor', 'handling',
                 'interior', 'exterior', 'design', 'metallic', 'pearl', 'blue', 'black', 'white', 'red',
                 'grey', 'midnight', 'cherry', 'autopilot', 'buy', 'sunroof', 'panomaric', 'glass roof',
                 'safety', 'street', 'highway'
                ]

In [18]:
filtered_df = df_tesla[df_tesla['Text'].str.lower().str.contains('|'.join(related_words))]

In [19]:
filtered_df

,Text,Datetime,Word,Make,Model
1,"CarlinKit Tesla Wireless CarPlay Adapter, for ...",2023-04-09 23:32:34+00:00,tesla,Tesla,not mentioned
2,@user @user @user The only problem is RIVN kee...,2023-04-09 23:26:38+00:00,tesla,Tesla,not mentioned
4,1074812-00-B New Set of (4) TPMS Tire Pressure...,2023-04-09 23:18:28+00:00,tesla,Tesla,not mentioned
9,@user @user @user My guests have great experie...,2023-04-09 23:02:19+00:00,tesla,Tesla,not mentioned
10,Tesla Model X Sinks In Danube River And Sells ...,2023-04-09 23:01:30+00:00,tesla,Tesla,not mentioned
...,...,...,...,...,...
244154,Watch Tesla CEO Elon Musk test drive VW ID.3 -...,2020-09-07 13:46:02+00:00,tesla,Tesla,not mentioned
244164,Is It Tesla Enough For You? Volkswagen Reveals...,2020-09-04 17:20:31+00:00,tesla,Tesla,not mentioned
244167,ID.4 Will Not Rival the Tesla Model Y #ev #ele...,2020-08-31 15:10:40+00:00,tesla,Tesla,not mentioned
244184,Volkswagen's ID.4 EV Crossover To Arrive In De...,2020-08-07 14:19:42+00:00,tesla,Tesla,not mentioned


In [20]:
# Removing rows that do not have related words
df.drop(df[(df['Make'] == 'Tesla') & (df['Model'] == 'not mentioned')].index, inplace=True)
df = pd.concat([df, filtered_df])

In [21]:
df

,Text,Datetime,Word,Make,Model
0,@user @user @user Definitely. R1S is a much be...,2023-04-09 23:44:47+00:00,model x,Tesla,Model X
3,"Model X Long Range Demo\nKing Of Prussia, PA\n...",2023-04-09 23:23:56+00:00,model x,Tesla,Model X
5,"Model X Long Range New\nBuena Park, CA\nPearl ...",2023-04-09 23:17:51+00:00,model x,Tesla,Model X
6,"Model X Long Range New\nCarlsbad, CA\nMidnight...",2023-04-09 23:17:50+00:00,model x,Tesla,Model X
7,"Model X Long Range Demo\nIndianapolis, IN\nSol...",2023-04-09 23:17:48+00:00,model x,Tesla,Model X
...,...,...,...,...,...
244154,Watch Tesla CEO Elon Musk test drive VW ID.3 -...,2020-09-07 13:46:02+00:00,tesla,Tesla,not mentioned
244164,Is It Tesla Enough For You? Volkswagen Reveals...,2020-09-04 17:20:31+00:00,tesla,Tesla,not mentioned
244167,ID.4 Will Not Rival the Tesla Model Y #ev #ele...,2020-08-31 15:10:40+00:00,tesla,Tesla,not mentioned
244184,Volkswagen's ID.4 EV Crossover To Arrive In De...,2020-08-07 14:19:42+00:00,tesla,Tesla,not mentioned


In [22]:
# Dropping amazon advertisements
df.drop(df[df['Text'].str.lower().str.contains('#amazon')].index, inplace=True)

In [24]:
#for tableau purposes
# df_excel = df.drop(df[df['Datetime'] >= '2023'].index)
# df_excel = df_excel.reset_index()
df.to_excel('../preprocesseddata/2020-01-01_2023-04-10.xlsx', index=False) 

In [25]:
df.to_csv('../preprocesseddata/2020-01-01_2023-04-10.csv', index=False)